## Training best performing model

In [ ]:
from util import *
import autograd.numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
# Load MNIST data
mnist = fetch_openml('mnist_784', version=1, parser='auto')
X = mnist.data.to_numpy() / 255.0
y = mnist.target.astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=10000 / 70000, random_state=42)
# One-hot encode
train_targets = one_hot_encode(y_train, num_classes=10)
test_targets = one_hot_encode(y_test, num_classes=10)

nn = NeuralNetwork(784, [100, 50, 10], [leaky_ReLU, leaky_ReLU, linear], [leaky_ReLU_der, leaky_ReLU_der, linear_der], cross_entropy_logits, cross_entropy_logits_der)
nn.gradient_descent_stochastic(X_train, train_targets, learning_rate=0.05, epochs=15, minibatch_size=100)
nn_pred = nn.predict_labels(X_test)

accuracy = accuracy_score(y_test, nn_pred)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

## Plotting confusion matrix

In [ ]:
# Confusion matrix
nn_pred = nn.predict_labels(X_test)

conf_matrix = np.zeros((10, 10), dtype=int)
for true_label, pred_label in zip(y_test, nn_pred):
	conf_matrix[true_label, pred_label] += 1

plt.figure(figsize=(10, 7))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap = "rocket")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("MNIST confusion matrix")
plt.show()

## Small guessing game for incorrectly classified numbers

In [ ]:
nn_pred = nn.predict_labels(X_test)

# Find all misclassified samples
misclassified = [(pred, true, x) for (pred, true), x in zip(zip(nn_pred, y_test), X_test) if pred != true]

# Shuffle the misclassified samples
np.random.shuffle(misclassified)

# Show them one by one
for pred, true, x in misclassified:
    plt.imshow(x.reshape(28, 28), cmap='gray')
    plt.axis('off')
    plt.show()
    guess = int(input())
    if guess == true:
        print("Correct!")
    else:
        print(f"Wrong! The correct label is {true}.")
    print(f"Model predicted: {pred}")


## Best PyTorch model configuration

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from util import ProgressBar
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

class MNISTNet(nn.Module):
    def __init__(self):
        super(MNISTNet, self).__init__()
        self.fc1 = nn.Linear(784, 200)
        self.fc2 = nn.Linear(200, 200)
        self.fc3 = nn.Linear(200, 10)
        self.leaky_relu = nn.LeakyReLU(negative_slope=0.01)
    
    def forward(self, x):
        x = self.leaky_relu(self.fc1(x))
        x = self.leaky_relu(self.fc2(x))
        x = self.fc3(x)
        return x

mnist = fetch_openml('mnist_784', version=1, parser='auto')
X = mnist.data.to_numpy() / 255.0
y = mnist.target.astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=10000 / 70000, random_state=42)

X_train_torch = torch.FloatTensor(X_train)
y_train_torch = torch.LongTensor(y_train.to_numpy())
X_test_torch = torch.FloatTensor(X_test)
y_test_torch = torch.LongTensor(y_test.to_numpy())

train_dataset = TensorDataset(X_train_torch, y_train_torch)
train_loader = DataLoader(train_dataset, batch_size=100, shuffle=True)

model = MNISTNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.RMSprop(model.parameters(), lr=0.0005)

epochs = 15
model.train()
bar = ProgressBar(epochs)
for epoch in range(epochs):    
    for batch_X, batch_y in train_loader:
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    bar.step()
bar.finish()

model.eval()
with torch.no_grad():
    outputs = model(X_test_torch)
    _, predicted = torch.max(outputs, 1)
    torch_accuracy = (predicted == y_test_torch).sum().item() / len(y_test_torch)
    print(f"\nPyTorch Test Accuracy: {torch_accuracy * 100:.2f}%")

### Confusion matrix for the best configuration

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
# Confusion matrix for PyTorch model
model.eval()
with torch.no_grad():
    outputs = model(X_test_torch)
    _, torch_pred = torch.max(outputs, 1)
    torch_pred_np = torch_pred.cpu().numpy()

conf_matrix_torch = np.zeros((10, 10), dtype=int)
for true_label, pred_label in zip(y_test, torch_pred_np):
    conf_matrix_torch[true_label, pred_label] += 1

plt.figure(figsize=(10, 7))
sns.heatmap(conf_matrix_torch, annot=True, fmt="d", cmap="rocket")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("PyTorch MNIST Confusion Matrix")
plt.show()